<a href="https://colab.research.google.com/github/kokami236/osiro1/blob/seikou2val/endewakeru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install open3d


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.2 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


点群の数の確認


In [ ]:
import open3d as o3d
import numpy as np

def count_points_with_open3d(file_path):
    # .plyファイルを読み込む
    print(f"ファイルを読み込んでいます: {file_path}")
    pcd = o3d.io.read_point_cloud(file_path)

    # 点群データが存在するか確認
    if pcd.is_empty():
        print("点群データが空、もしくはファイルの読み込みに失敗しました。")
        return

    # 点群の数を取得 (numpy配列に変換して形状を確認、またはlen()を使用)
    point_count = len(pcd.points)

    print(f"--- 結果 ---")
    print(f"点群の数: {point_count}")

    # 必要であればnumpy配列として詳細データを取得可能
    # points_np = np.asarray(pcd.points)
    # print(points_np.shape)

# ファイルパスを指定して実行
file_path = "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply"  # ここに読み込みたいファイル名を指定
count_points_with_open3d(file_path)

ファイルを読み込んでいます: /content/drive/MyDrive/ジオラマ/熊本正解データ.ply
--- 結果 ---
点群の数: 970651


上ではただ円で切り分けるだけ、ダウンサンプリングなし
したコードをそのあとに適応する


In [ ]:
import open3d as o3d
import numpy as np
import os

# ==========================================
# --- ユーザー設定エリア ---
# ==========================================

# 1. 入力ファイル
input_file = "/content/drive/MyDrive/ジオラマ/小倉城out1.ply"

# 2. 保存ファイル
output_file = "/content/drive/MyDrive/ジオラマ/小倉城out2.ply"

# 3. 残したい半径 (メートル単位)
# ここで設定した半径より外側をカットします
radius = 1.7

# ==========================================
# --- 処理実行エリア ---
# ==========================================

if not os.path.exists(input_file):
    print(f"エラー: 入力ファイルが見つかりません: {input_file}")
else:
    # 1. ファイル読み込み
    print("ファイルを読み込んでいます...")
    pcd = o3d.io.read_point_cloud(input_file)

    if not pcd.has_points():
        print("エラー: 点が含まれていません。")
    else:
        original_count = len(pcd.points)
        print(f"元の点の数: {original_count}")

        # ------------------------------------------
        # 2. 重心の計算 (元の高密度なデータから計算)
        # ------------------------------------------
        center = pcd.get_center()
        print(f"データの重心: {center}")

        # ------------------------------------------
        # 3. 半径による範囲カット
        # ------------------------------------------
        print(f"半径 {radius}m 以内の点を抽出中...")

        # 各点と重心との距離を計算
        points = np.asarray(pcd.points)
        distances = np.linalg.norm(points - center, axis=1)

        # 指定半径(radius)以内の点だけを選ぶ
        indices = np.where(distances <= radius)[0]

        # 抽出を実行
        final_pcd = pcd.select_by_index(indices)

        # ------------------------------------------
        # 4. 結果の保存
        # ------------------------------------------
        final_count = len(final_pcd.points)

        # 削除された点の割合を計算
        if original_count > 0:
            reduction_rate = (1 - final_count / original_count) * 100
        else:
            reduction_rate = 0

        print(f"最終的に残った点の数: {final_count}")
        print(f"カットされた割合: {reduction_rate:.2f}%")

        o3d.io.write_point_cloud(output_file, final_pcd)
        print(f"保存完了: {output_file}")

ファイルを読み込んでいます...
元の点の数: 1073984
データの重心: [-0.18435264 -0.60698334 -0.20332932]
半径 1.7m 以内の点を抽出中...
最終的に残った点の数: 957301
カットされた割合: 10.86%
保存完了: /content/drive/MyDrive/ジオラマ/小倉城out2.ply


In [ ]:
import open3d as o3d
import numpy as np
import os

# ==========================================
# --- ユーザー設定エリア ---
# ==========================================

# 1. 入力ファイル
input_file = "/content/drive/MyDrive/ジオラマ/小倉城out1.ply"

# 2. 保存ファイル
output_file = "/content/drive/MyDrive/ジオラマ/小倉城out2.ply"

# 3. ボクセルサイズ (軽量化の度合い)
voxel_size = 0.01  # 例: 0.01 = 1cm間隔

# 4. 残したい半径 (メートル単位)
# 軽量化した後の重心から、この半径より外側をカットします
radius = 1.9

# ==========================================
# --- 処理実行エリア ---
# ==========================================

if not os.path.exists(input_file):
    print(f"エラー: 入力ファイルが見つかりません: {input_file}")
else:
    # 1. ファイル読み込み
    print("ファイルを読み込んでいます...")
    pcd = o3d.io.read_point_cloud(input_file)

    if not pcd.has_points():
        print("エラー: 点が含まれていません。")
    else:
        print(f"元の点の数: {len(pcd.points)}")

        # ------------------------------------------
        # 2. ボクセルダウンサンプリング（軽量化）
        # ------------------------------------------
        print(f"ボクセルサイズ {voxel_size} でダウンサンプリング中...")
        downsampled_pcd = pcd.voxel_down_sample(voxel_size=voxel_size)

        # 軽くなった時点での点数を表示
        temp_count = len(downsampled_pcd.points)
        print(f"ダウンサンプリング後の点の数: {temp_count}")

        if temp_count == 0:
            print("警告: 点がなくなりました。voxel_sizeを小さくしてください。")
        else:
            # ------------------------------------------
            # 3. 重心の再計算 & ノイズ削除
            # ------------------------------------------

            # ★ポイント: 「軽量化された点群(downsampled_pcd)」の重心を計算します
            center = downsampled_pcd.get_center()
            print(f"軽量化後のデータから計算した重心: {center}")

            # 各点と重心との距離を計算
            points = np.asarray(downsampled_pcd.points)
            distances = np.linalg.norm(points - center, axis=1)

            # 指定半径(radius)以内の点だけを選ぶ
            indices = np.where(distances <= radius)[0]
            final_pcd = downsampled_pcd.select_by_index(indices)

            # ------------------------------------------
            # 4. 結果の保存
            # ------------------------------------------
            final_count = len(final_pcd.points)
            reduction_rate = (1 - final_count / len(pcd.points)) * 100

            print(f"最終的に残った点の数: {final_count}")
            print(f"トータルの削減率: {reduction_rate:.2f}%")

            o3d.io.write_point_cloud(output_file, final_pcd)
            print(f"保存完了: {output_file}")

ファイルを読み込んでいます...
元の点の数: 1073984
ボクセルサイズ 0.01 でダウンサンプリング中...
ダウンサンプリング後の点の数: 252286
軽量化後のデータから計算した重心: [-0.22283563 -0.73264615 -0.24975408]
最終的に残った点の数: 230667
トータルの削減率: 78.52%
保存完了: /content/drive/MyDrive/ジオラマ/小倉城out2.ply


In [ ]:
import open3d as o3d
import numpy as np
import os

# ==========================================
# --- 設定エリア（ステップ3） ---
# ==========================================

# 1. 入力ファイル
# 前のステップ（軽量化）で保存したファイルを指定します
input_file = "/content/drive/MyDrive/ジオラマ/小倉城out2.ply"

# 2. 最終的に残したい半径
# 軽量化によって重心位置がわずかに変わっている可能性があるため、
# ここで設定した半径で再度きれいに切り抜きます。
radius = 2.22

# 3. 保存するファイル名（完成データ）
output_file = "/content/drive/MyDrive/ジオラマ/小倉城_最終完成.ply"

# ==========================================
# --- 処理実行 ---
# ==========================================

if not os.path.exists(input_file):
    print(f"エラー: 前のステップの出力ファイルが見つかりません: {input_file}")
    print("一つ前のコードセル（軽量化処理）が正しく実行されているか確認してください。")
else:
    # ファイル読み込み
    pcd = o3d.io.read_point_cloud(input_file)

    if not pcd.has_points():
        print("エラー: 点群データが空です。")
    else:
        print(f"軽量化済みデータの点数: {len(pcd.points)}")

        # --- 重心の再計算 ---
        # 軽くなったデータに基づいて、改めて重心を求めます
        center = pcd.get_center()
        print(f"再計算された重心座標: {center}")

        # --- 距離によるフィルタリング（円形/球形カット） ---
        points = np.asarray(pcd.points)
        distances = np.linalg.norm(points - center, axis=1)

        # 半径以内のインデックスを取得
        indices = np.where(distances <= radius)[0]

        # 抽出実行
        final_pcd = pcd.select_by_index(indices)

        # --- 結果表示と保存 ---
        final_count = len(final_pcd.points)
        removed_count = len(pcd.points) - final_count

        print(f"半径 {radius}m 以内に残った点数: {final_count}")
        print(f"削除されたノイズ（外側の点）の数: {removed_count}")

        o3d.io.write_point_cloud(output_file, final_pcd)
        print(f"全ての処理が完了しました。保存先: {output_file}")

In [ ]:
import open3d as o3d
import numpy as np
import os

# --- ユーザーが設定する項目 ---

# 1. 入力する点群ファイルのパス
input_file = "/content/drive/MyDrive/熊本城外きれい1.ply"

# 2. 中心の点から残したい半径 (単位は点群の座標系に依存します)
# この値を大きくすると、より多くの点が残ります。
radius = 3.0

# 3. 保存するファイル名
output_file = "/content/drive/MyDrive/centered_point_cloud.ply"

# --------------------------


# ファイルの存在を確認
if not os.path.exists(input_file):
    print(f"エラー: 入力ファイルが見つかりません: {input_file}")
else:
    # 点群データを読み込む
    pcd = o3d.io.read_point_cloud(input_file)

    if not pcd.has_points():
        print("エラー: 点群の読み込みに失敗したか、点が含まれていません。")
    else:
        print(f"処理前の点の数: {len(pcd.points)}")

        # 1. 点群の重心（中心）を計算
        center = pcd.get_center()
        print(f"計算された中心座標: {center}")

        # 2. 各点が中心からどれだけ離れているか計算
        points = np.asarray(pcd.points)

        # NumPyを使って全点の中心からの距離を高速に計算
        distances = np.linalg.norm(points - center, axis=1)

        # 3. 指定した半径の内側にある点のインデックスを取得
        indices = np.where(distances <= radius)[0]

        # 4. 半径内の点群だけを抽出して新しい点群オブジェクトを作成
        centered_pcd = pcd.select_by_index(indices)

        print(f"半径 {radius} m 内の点の数: {len(centered_pcd.points)}")

        # 5. 結果をファイルに書き出す
        o3d.io.write_point_cloud(output_file, centered_pcd)
        print(f"処理後のファイルを {output_file} に保存しました。")

処理前の点の数: 1093108
計算された中心座標: [ 0.59900511 -0.193925    0.99813606]
半径 3.0 m 内の点の数: 1029450
処理後のファイルを /content/drive/MyDrive/centered_point_cloud.ply に保存しました。
